In [ ]:
# %%
import os
import numpy as np
import h5py

rng = np.random.default_rng(7)

file = 'CSTR_InputVectors_Unseen.h5'   # output file

N = 100_000
N_CONTEXT = 75_000              # first 75% = context region: only "seen" faults, one at a time
t = np.arange(N, dtype=float)

# same physical definitions as CSTR-Input.ipynb; sign is set per event below
FAULTS = {
    1: dict(name='E_R',    baseline=8750.0,  delta=270.0,       variance=0.1),
    2: dict(name='U_Ac',   baseline=5e4/60,  delta=-225.0,      variance=0.1),
    3: dict(name='T_Bias', baseline=0.0,     delta=5.0,         variance=0.5),
    4: dict(name='T_F',    baseline=320.0,   delta=0.35*90*2.5, variance=0.2),
    5: dict(name='C_F',    baseline=1.0,     delta=0.0006*90*2.5, variance=0.3),
    6: dict(name='T_CF',   baseline=300.0,   delta=0.1*150*2,   variance=0.2),
    7: dict(name='Q_F',    baseline=100/60,  delta=10/60*3,     variance=0.1),
    8: dict(name='dP',     baseline=50.0,    delta=5.0,         variance=0.4),
    9: dict(name='dPc',    baseline=25.0,    delta=2.5,         variance=0.4),
}

# --- Context region: F1 (E_R) and F7 (Q_F) never occur, F4 (T_F) only ever +, F2/F5/F6 as usual ---
CONTEXT_EVENTS = {   # fid: list of signs, one per occurrence
    2: [1, 1, 1, 1],
    4: [1, 1, 1, 1],
    5: [1, 1, -1, -1],
    6: [1, 1, -1, -1],
}
DURATION_RANGE = (2000, 4000)
RAMP_FRACTION = 0.15
MIN_GAP = 300
SETTLE_BEFORE_TEST = 1500       # baseline samples at the end of the context region, so the test region starts settled

# --- Test region: 6 cases, each novel element appears exactly once so earlier cases give nothing away about later ones ---
# (fid, sign) per fault; two entries = overlapping faults, the second starting OVERLAP_OFFSET into the first
TEST_CASES = [
    ('unseen E_R',              [(1, 1)]),
    ('overlap U_Ac + C_F-',     [(2, 1), (5, -1)]),
    ('opposite direction T_F-', [(4, -1)]),
    ('overlap T_F+ + T_CF-',    [(4, 1), (6, -1)]),
    ('unseen Q_F (+)',          [(7, 1)]),
    ('overlap C_F+ + T_CF+',    [(5, 1), (6, 1)]),
]
TEST_DURATION_RANGE = (2000, 2400)   # shorter end of the context range so 6 cases + settling gaps fit in the last 25%
OVERLAP_OFFSET_RANGE = (700, 1000)

vectors = {fid: cfg['baseline'] * np.ones(N) for fid, cfg in FAULTS.items()}
flags = {fid: np.zeros(N) for fid in FAULTS}
holds = {fid: np.zeros(N) for fid in FAULTS}
event_info = []   # (region, case, fid, start, end, up_dur, hold_dur, down_dur, magnitude)

def add_event(fid, sign, start, dur, region, case):
    cfg = FAULTS[fid]
    magnitude = sign * cfg['delta'] * (1 + rng.uniform(0, cfg['variance']))
    up_dur = down_dur = max(1, round(dur * RAMP_FRACTION))
    hold_dur = dur - up_dur - down_dur
    perturbation = np.concatenate([np.linspace(0, magnitude, up_dur), np.full(hold_dur, magnitude), np.linspace(magnitude, 0, down_dur)])
    vectors[fid][start:start + dur] = cfg['baseline'] + perturbation
    flags[fid][start:start + dur] = 1
    holds[fid][start + up_dur:start + up_dur + hold_dur] = 1
    event_info.append((region, case, fid, start, start + dur, up_dur, hold_dur, down_dur, magnitude))

# context schedule: same strict single-active-fault Dirichlet gap placement as CSTR-Input.ipynb
events = [(fid, s) for fid, signs in CONTEXT_EVENTS.items() for s in signs]
rng.shuffle(events)
durations = rng.integers(DURATION_RANGE[0], DURATION_RANGE[1] + 1, size=len(events))
span = N_CONTEXT - SETTLE_BEFORE_TEST
steady = span - int(durations.sum())
num_gaps = len(events) + 1
if steady < MIN_GAP * num_gaps:
    raise ValueError("Not enough room in the context region for the requested faults.")
gaps = np.floor(rng.dirichlet(np.ones(num_gaps)) * (steady - MIN_GAP * num_gaps)).astype(int) + MIN_GAP
gaps[-1] += steady - gaps.sum()

pos = gaps[0]
for i, (fid, sign) in enumerate(events):
    add_event(fid, sign, pos, int(durations[i]), 'context', '')
    pos += int(durations[i]) + gaps[i + 1]
assert pos == span

# test schedule: equal settling gaps between cases, with the leftover at the end so the last case's recovery is visible
case_plan = []
for name, faults in TEST_CASES:
    durs = rng.integers(TEST_DURATION_RANGE[0], TEST_DURATION_RANGE[1] + 1, size=len(faults))
    offsets = [0] + [int(rng.integers(*OVERLAP_OFFSET_RANGE)) for _ in faults[1:]]
    case_plan.append((name, faults, durs, offsets, max(o + d for o, d in zip(offsets, durs))))
test_gap = (N - N_CONTEXT - sum(p[-1] for p in case_plan)) // (len(case_plan) + 1)
if test_gap < 1000:
    raise ValueError(f"Test gaps of {test_gap} samples are too short for the system to settle between cases.")

pos = N_CONTEXT + test_gap
for name, faults, durs, offsets, case_span in case_plan:
    for (fid, sign), dur, off in zip(faults, durs, offsets):
        add_event(fid, sign, pos + off, int(dur), 'test', name)
    pos += case_span + test_gap

# AR1 measurement noise, applied to F3-F9 (identical to CSTR-Input.ipynb)
def AR1(n, p, s, rng):
    ar = np.zeros(n)
    for i in range(1, n):
        ar[i] = ar[i - 1] * p + rng.normal(0, s)
    return ar / 100 + 1

p, s = 0.9999, 0.005
for fid in (3, 4, 5, 6, 7, 8, 9):
    vectors[fid] = vectors[fid] * AR1(N, p, s, rng)

# --- Checks ------------------------------------------------------------------
overlap_count = np.vstack(list(flags.values())).sum(axis=0)
assert overlap_count[:N_CONTEXT].max() <= 1, "Faults overlap in the context region"
assert flags[1][:N_CONTEXT].sum() == 0 and flags[7][:N_CONTEXT].sum() == 0, "Unseen fault leaked into the context region"
assert all(m > 0 for r, _, fid, *_, m in event_info if r == 'context' and fid == 4), "T_F- leaked into the context region"
assert overlap_count[N_CONTEXT - SETTLE_BEFORE_TEST:N_CONTEXT].max() == 0

print(f"Context region [0, {N_CONTEXT}): {sum(r == 'context' for r, *_ in event_info)} events, "
      f"{(overlap_count[:N_CONTEXT] == 0).mean():.1%} at nominal baseline")
for fid in CONTEXT_EVENTS:
    info = [e for e in event_info if e[0] == 'context' and e[2] == fid]
    mags = [e[-1] for e in info]
    print(f"  F{fid} ({FAULTS[fid]['name']}): {len(info)} events, signs {[int(np.sign(m)) for m in mags]}, "
          f"magnitudes {min(mags):.4g} .. {max(mags):.4g}")
print(f"Test region [{N_CONTEXT}, {N}): {len(TEST_CASES)} cases, settling gap {test_gap} samples between cases")
for name, *_ in case_plan:
    info = [e for e in event_info if e[1] == name]
    desc = ', '.join(f"F{fid} {FAULTS[fid]['name']} [{st}, {en}) mag {m:+.4g}" for _, _, fid, st, en, *_, m in info)
    print(f"  {name}: {desc}")

# --- Save --------------------------------------------------------------------
def save_vector(filename, name, data):
    with h5py.File(filename, 'a') as f:
        if name in f:
            del f[name]
        f.create_dataset(name, data=data, chunks=True, compression='gzip')
        print(f"Saved dataset '{name}' with shape {data.shape}")

if os.path.exists(file):
    os.remove(file)

save_vector(file, 't', t)
for fid in FAULTS:
    save_vector(file, f'F{fid}', vectors[fid].reshape(-1, 1))
    save_vector(file, f'F{fid}.plt', flags[fid])